# 예제 04. pandas에서 Tensor로
빅데이터프로그래밍 · 3주차

## 목표
- DataFrame → NumPy → Tensor 로 변환한다
- `float32` 로 맞춘다
- 배치 단위로 꺼내 본다 (DataLoader)

pandas는 확인과 정리, Tensor는 계산. 학습 직전에 한 번 변환합니다.


In [ ]:
csv = """name,gender,department,study_hours,attendance,midterm,final
김통계,남,통계학과,12.5,95%,88,92
이확률,여,통계학과,8.0,88%,92,85
박회귀,남,컴퓨터공학과,,72%,79,68
최추정,여,통계학과,15.0,100%,95,98
정검정,남,경제학과,4.5,61%,61,
한분산,여,컴퓨터공학과,10.0,90%,84,88
오평균,남,경제학과,6.5,,70,74
서표본,여,통계학과,13.0,97%,100,96
남표준,남,컴퓨터공학과,9.5,85%,66,71
윤편차,여,경제학과,,80%,82,79
"""

with open("students.csv", "w", encoding="utf-8") as f:
    f.write(csv)

print("students.csv 생성 완료")


In [ ]:
import pandas as pd
import numpy as np
import torch

df = pd.read_csv("students.csv")

df["study_hours"] = df["study_hours"].fillna(df["study_hours"].mean())
df["final"] = df["final"].fillna(df["final"].mean())
df["attendance"] = df["attendance"].str.replace("%", "", regex=False).astype(float)
df["attendance"] = df["attendance"].fillna(df["attendance"].median())
df["gender"] = df["gender"].map({"남": 0, "여": 1})
df = pd.get_dummies(df, columns=["department"], dtype=int).drop(columns=["name"])
df["pass"] = (df["final"] >= 80).astype(int)

X_df = df.drop(columns=["pass", "final"])
y_df = df["pass"]
X_df.head(3)


## 1. DataFrame → NumPy


In [ ]:
X_np = X_df.to_numpy(dtype="float32")
y_np = y_df.to_numpy(dtype="float32")

print(type(X_np), X_np.shape, X_np.dtype)
print(type(y_np), y_np.shape, y_np.dtype)


## 2. NumPy → Tensor
딥러닝 계산은 `float32` 가 기본입니다. `float64` 를 넣으면 자료형 오류가 납니다.


In [ ]:
X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print(X[:2])


In [ ]:
# 정답은 보통 열 벡터 (n, 1) 로 만듭니다
y = y.reshape(-1, 1)
print("y:", y.shape)


## 3. 전처리 결과 확인 — 학습 전 다섯 가지


In [ ]:
print("데이터 개수 :", X.shape[0])
print("입력 shape  :", tuple(X.shape))
print("자료형      :", X.dtype)
print("최솟값/최댓값:", X.min().item(), "/", X.max().item())
print("결측(NaN) 수 :", torch.isnan(X).sum().item())


## 4. 값의 범위가 크게 다르면 학습이 불안정합니다


In [ ]:
for i, name in enumerate(X_df.columns):
    col = X[:, i]
    print(f"{name:22s} min {col.min():7.2f}  max {col.max():7.2f}")


In [ ]:
# 표준화 후 다시 확인
mean = X.mean(dim=0)
std = X.std(dim=0)
std[std == 0] = 1.0          # 상수 열 보호

Xs = (X - mean) / std
print("표준화 후 min/max:", Xs.min().item().__round__(2), "/", Xs.max().item().__round__(2))


## 5. 배치로 꺼내기 — DataLoader 맛보기
학습은 전체를 한 번에 넣지 않고 작은 묶음(배치)으로 나눠 진행합니다. 6주차에 본격적으로 씁니다.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(Xs, y)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

print("전체 개수:", len(dataset))

for bx, by in loader:
    print("배치 X:", tuple(bx.shape), "배치 y:", tuple(by.shape))


## 직접 해보기
1. `batch_size` 를 3으로 바꾸면 마지막 배치의 크기는 몇인가요?
2. `X.double()` 로 자료형을 바꾼 뒤 shape과 dtype을 출력하세요.


In [ ]:
# 여기에 작성하세요
